In [1]:
import os
import re

def check_anomaly_in_file(file_path):
    anomalies = []
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        if not lines:
            anomalies.append((0, "파일이 비어 있습니다."))
            return anomalies
        
        for i, line in enumerate(lines):
            line = line.strip()
            # 빈 줄 또는 의미 없는 문자열만 있는 경우
            if not line or re.match(r'^[\s.,;]*$', line):
                anomalies.append((i + 1, f"빈 줄 또는 의미 없는 문자열: '{line}'"))
                continue
            
            # 클래스 ID와 바운딩 박스 검사
            parts = line.split()
            if len(parts) != 5:
                anomalies.append((i + 1, f"잘못된 형식: '{line}'"))
                continue
            
            try:
                class_id = int(parts[0])
                bbox = list(map(float, parts[1:]))
                if class_id < 0 or class_id >= 3:  # 클래스 개수가 3개이므로 0, 1, 2만 허용
                    anomalies.append((i + 1, f"잘못된 클래스 ID: '{class_id}'"))
                if any(x < 0 or x > 1 for x in bbox):
                    anomalies.append((i + 1, f"바운딩 박스 범위 초과: '{bbox}'"))
            except ValueError:
                anomalies.append((i + 1, f"형식 오류: '{line}'"))
    
    return anomalies

def find_anomalies_in_directory(directory):
    anomaly_files = {}
    total_anomalies = 0
    
    for root, _, files in os.walk(directory):
        for file in files:
            if file.endswith('.txt'):
                file_path = os.path.join(root, file)
                anomalies = check_anomaly_in_file(file_path)
                if anomalies:
                    anomaly_files[file_path] = anomalies
                    total_anomalies += len(anomalies)
    
    return anomaly_files, total_anomalies

def main():
    # 데이터셋 기본 경로 설정
    DATASET_BASE_DIR = r'C:\Users\AREU\Desktop\PolyProject\animal\YOLODataset'
    
    # 검사할 폴더 목록
    folders_to_check = [
        os.path.join(DATASET_BASE_DIR, 'labels', 'train'),
        os.path.join(DATASET_BASE_DIR, 'labels', 'val')
    ]
    
    total_anomalies = 0
    
    for folder in folders_to_check:
        print(f"검사 중인 폴더: {folder}")
        anomaly_files, folder_anomalies = find_anomalies_in_directory(folder)
        total_anomalies += folder_anomalies
        
        if anomaly_files:
            print(f"이상치 데이터 발견: {folder_anomalies} 개")
            for file_path, anomalies in anomaly_files.items():
                print(f"파일: {file_path}")
                for line_num, message in anomalies:
                    print(f"  줄 {line_num}: {message}")
        else:
            print("이상치 데이터 없음")
        print("-" * 30)
    
    print(f"총 이상치 데이터 개수: {total_anomalies}")

if __name__ == "__main__":
    main()

검사 중인 폴더: C:\Users\AREU\Desktop\PolyProject\animal\YOLODataset\labels\train
이상치 데이터 발견: 738 개
파일: C:\Users\AREU\Desktop\PolyProject\animal\YOLODataset\labels\train\A01_F03_C074_C_211112_1003_20S_000004.005.txt
  줄 0: 파일이 비어 있습니다.
파일: C:\Users\AREU\Desktop\PolyProject\animal\YOLODataset\labels\train\A01_F03_C074_C_211112_1005_20S_000016.030.txt
  줄 0: 파일이 비어 있습니다.
파일: C:\Users\AREU\Desktop\PolyProject\animal\YOLODataset\labels\train\A01_G12_G001_G_200318_4019_39S_000037.783.txt
  줄 0: 파일이 비어 있습니다.
파일: C:\Users\AREU\Desktop\PolyProject\animal\YOLODataset\labels\train\A01_G12_G001_G_200523_4023_18S_000012.406.txt
  줄 0: 파일이 비어 있습니다.
파일: C:\Users\AREU\Desktop\PolyProject\animal\YOLODataset\labels\train\A01_G22_C024_D_211118_2033_59S_000049.254.txt
  줄 1: 바운딩 박스 범위 초과: '[0.194497, 1.118273, 0.34613, 0.683194]'
파일: C:\Users\AREU\Desktop\PolyProject\animal\YOLODataset\labels\train\A01_H29_C005_C_211019_2001_20S_000010.303.txt
  줄 0: 파일이 비어 있습니다.
파일: C:\Users\AREU\Desktop\PolyProject\animal\YO

In [1]:
import os
import re
import shutil
import zipfile
from datetime import datetime

def process_anomaly_in_file(file_path):
    """
    파일에서 이상치를 찾고 처리합니다.
    - 비어있는 파일은 삭제합니다.
    - 형식 오류, 바운딩 박스 범위 초과, 클래스 오류가 있는 라인은 삭제합니다.
    - 모든 라인이 삭제되어 비어있게 된 파일도 삭제합니다.
    
    Returns:
        bool: 파일이 삭제되었으면 True, 아니면 False
    """
    # 파일이 비어있는지 확인
    if os.path.getsize(file_path) == 0:
        os.remove(file_path)
        return True
    
    modified = False
    valid_lines = []
    
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        
        for line in lines:
            line = line.strip()
            # 빈 줄 또는 의미 없는 문자열만 있는 경우 스킵
            if not line or re.match(r'^[\s.,;]*$', line):
                modified = True
                continue
            
            # 클래스 ID와 바운딩 박스 검사
            parts = line.split()
            if len(parts) != 5:
                modified = True
                continue
            
            try:
                class_id = int(parts[0])
                bbox = list(map(float, parts[1:]))
                
                # 클래스 ID나 바운딩 박스 값이 범위를 벗어나면 스킵
                if class_id < 0 or class_id >= 3 or any(x < 0 or x > 1 for x in bbox):
                    modified = True
                    continue
                
                # 유효한 라인만 저장
                valid_lines.append(line)
            except ValueError:
                modified = True
                continue
    
    # 유효한 라인이 없다면 파일 삭제
    if not valid_lines:
        os.remove(file_path)
        return True
    
    # 수정된 내용이 있다면 파일 다시 쓰기
    if modified:
        with open(file_path, 'w', encoding='utf-8') as f:
            for line in valid_lines:
                f.write(line + '\n')
    
    return False

def process_anomalies_in_directory(directory, deleted_files):
    """
    디렉토리 내의 모든 txt 파일에서 이상치를 처리하고 삭제된 파일 목록을 반환합니다.
    """
    for root, _, files in os.walk(directory):
        for file in files:
            if file.endswith('.txt'):
                file_path = os.path.join(root, file)
                is_deleted = process_anomaly_in_file(file_path)
                if is_deleted:
                    deleted_files.append(file_path)
    
    return deleted_files

def backup_deleted_files(deleted_files, backup_dir):
    """
    삭제된 파일들을 백업 디렉토리에 복사하고 zip으로 압축합니다.
    """
    if not deleted_files:
        print("삭제된 파일이 없습니다.")
        return None
    
    # 백업 디렉토리 생성
    os.makedirs(backup_dir, exist_ok=True)
    
    # 현재 시간을 포함한 zip 파일 이름 생성
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    zip_filename = os.path.join(backup_dir, f"deleted_files_{timestamp}.zip")
    
    # 삭제된 파일들을 zip 파일로 압축
    with zipfile.ZipFile(zip_filename, 'w') as zipf:
        for file_path in deleted_files:
            # 이미 삭제된 파일이므로 백업할 수 없음
            if os.path.exists(file_path):
                arcname = os.path.basename(file_path)
                zipf.write(file_path, arcname)
                print(f"파일 백업: {file_path}")
    
    print(f"삭제된 파일 백업 완료: {zip_filename}")
    return zip_filename

def main():
    # 데이터셋 기본 경로 설정
    DATASET_BASE_DIR = r'C:\Users\AREU\Desktop\PolyProject\animal\YOLODataset'
    
    # 검사할 폴더 목록
    folders_to_check = [
        os.path.join(DATASET_BASE_DIR, 'labels', 'train'),
        os.path.join(DATASET_BASE_DIR, 'labels', 'val')
    ]
    
    # 백업 디렉토리 설정
    backup_dir = os.path.join(DATASET_BASE_DIR, 'deleted_backups')
    
    # 원본 파일 백업을 위한 임시 디렉토리
    temp_backup_dir = os.path.join(DATASET_BASE_DIR, 'temp_backup')
    os.makedirs(temp_backup_dir, exist_ok=True)
    
    deleted_files = []
    
    # 원본 파일 백업 (실제 삭제 전)
    for folder in folders_to_check:
        print(f"폴더 처리 중: {folder}")
        for root, _, files in os.walk(folder):
            for file in files:
                if file.endswith('.txt'):
                    src_path = os.path.join(root, file)
                    dst_path = os.path.join(temp_backup_dir, file)
                    shutil.copy2(src_path, dst_path)
    
    # 모든 폴더에서 이상치 처리
    for folder in folders_to_check:
        deleted_files = process_anomalies_in_directory(folder, deleted_files)
    
    # 삭제된 파일 개수 출력
    print(f"총 {len(deleted_files)}개 파일 삭제됨")
    
    # 삭제된 파일들의 백업 복사본을 zip으로 압축
    zip_file = backup_deleted_files([os.path.join(temp_backup_dir, os.path.basename(f)) for f in deleted_files], backup_dir)
    
    # 임시 백업 디렉토리 삭제
    shutil.rmtree(temp_backup_dir)
    
    if zip_file:
        print(f"삭제된 파일들이 {zip_file}에 백업되었습니다.")

if __name__ == "__main__":
    main()

폴더 처리 중: C:\Users\AREU\Desktop\PolyProject\animal\YOLODataset\labels\train
폴더 처리 중: C:\Users\AREU\Desktop\PolyProject\animal\YOLODataset\labels\val
총 834개 파일 삭제됨
파일 백업: C:\Users\AREU\Desktop\PolyProject\animal\YOLODataset\temp_backup\A01_F03_C074_C_211112_1003_20S_000004.005.txt
파일 백업: C:\Users\AREU\Desktop\PolyProject\animal\YOLODataset\temp_backup\A01_F03_C074_C_211112_1005_20S_000016.030.txt
파일 백업: C:\Users\AREU\Desktop\PolyProject\animal\YOLODataset\temp_backup\A01_G12_G001_G_200318_4019_39S_000037.783.txt
파일 백업: C:\Users\AREU\Desktop\PolyProject\animal\YOLODataset\temp_backup\A01_G12_G001_G_200523_4023_18S_000012.406.txt
파일 백업: C:\Users\AREU\Desktop\PolyProject\animal\YOLODataset\temp_backup\A01_G22_C024_D_211118_2033_59S_000049.254.txt
파일 백업: C:\Users\AREU\Desktop\PolyProject\animal\YOLODataset\temp_backup\A01_H29_C005_C_211019_2001_20S_000010.303.txt
파일 백업: C:\Users\AREU\Desktop\PolyProject\animal\YOLODataset\temp_backup\A01_H29_C005_C_211019_2002_20S_000001.562.txt
파일 백업: C:\Us